<a href="https://colab.research.google.com/github/MIDHUN-4126/24ADC001-24BAD069/blob/main/Experiment_4_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# EXPERIMENT NO. 4 — TRANSFER LEARNING USING PRE-TRAINED VISION MODELS FOR IMAGE RECOGNITION

**Name:** Midhun P
**Roll No.:** 24BAD069  

### Aim
To implement transfer learning using pre-trained vision models for image recognition and evaluate their performance on a real-world image dataset.




## Install and Import Required Libraries

This notebook uses TensorFlow/Keras for transfer learning, Matplotlib for visualization, and Hugging Face Transformers for pre-trained vision-model inference.


In [ ]:

!pip -q install -U transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
# drive.mount('/content/drive') # Commented out to prevent mount errors if not needed.

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0



## Task A — Dataset Preparation




In [ ]:
# If your Kaggle dataset is already downloaded, set its extracted folder here.
DATASET_DIR = "/content/archive.zip"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT = 0.20
TEST_SPLIT = 0.10

if not os.path.exists(DATASET_DIR):
    print("Dataset folder not found.")
    print("Upload/extract your Kaggle image dataset to:", DATASET_DIR)
else:
    print("Dataset found:", DATASET_DIR)


Dataset folder not found.
Upload/extract your Kaggle image dataset to: /content/archive.zip


In [ ]:
import zipfile

# Unzip the dataset if it's a zip file
initial_dataset_dir = DATASET_DIR # Store the initial path to the zip file

if initial_dataset_dir.endswith('.zip') and os.path.exists(initial_dataset_dir):
    # Define the directory where the content will be unzipped
    unzip_dir = initial_dataset_dir.replace('.zip', '') # Remove .zip extension for the directory name

    # Check if the directory already exists to avoid re-unzipping
    if not os.path.exists(unzip_dir):
        print(f"Unzipping {initial_dataset_dir} to {unzip_dir}...")
        with zipfile.ZipFile(initial_dataset_dir, 'r') as zip_ref:
            zip_ref.extractall(unzip_dir)
        print("Unzipping complete.")
    else:
        print(f"Directory {unzip_dir} already exists. Skipping unzipping.")

    # Update DATASET_DIR to point to the unzipped directory
    DATASET_DIR = unzip_dir

print("Current DATASET_DIR after unzipping:", DATASET_DIR)


Current DATASET_DIR after unzipping: /content/archive.zip


In [ ]:
# After unzipping, update DATASET_DIR to point to the 'train' subdirectory
# This is where the actual class folders (e.g., 'daisy', 'dandelion') are located.
DATASET_DIR = os.path.join(DATASET_DIR, 'train')
print("Final DATASET_DIR for dataset loading:", DATASET_DIR)

Final DATASET_DIR for dataset loading: /content/archive.zip/train


Let's inspect the contents of the `DATASET_DIR` to understand its structure, as the model is currently only detecting one class. For `tf.keras.utils.image_dataset_from_directory` to work correctly, your images should be organized into subdirectories, where each subdirectory name represents a class.

In [ ]:
import os

# List contents of the DATASET_DIR
print(f"Contents of {DATASET_DIR}:")
for item in os.listdir(DATASET_DIR):
    item_path = os.path.join(DATASET_DIR, item)
    if os.path.isdir(item_path):
        print(f"  - Directory: {item} (contains {len(os.listdir(item_path))} items)")
    else:
        print(f"  - File: {item}")

print("\nExpected structure: DATASET_DIR/class_name_1/image1.jpg, DATASET_DIR/class_name_2/image2.jpg")
print("If all images are directly under DATASET_DIR or a single subdirectory, `image_dataset_from_directory` will only detect one class.")

Contents of /content/archive.zip/train:


FileNotFoundError: [Errno 2] No such file or directory: '/content/archive.zip/train'

In [ ]:

# Create training and temporary validation/test datasets.
# The temporary set is later divided into validation and test sets.

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VAL_SPLIT + TEST_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VAL_SPLIT + TEST_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)


In [ ]:

# Split the temporary dataset into validation and test sets.
temp_batches = tf.data.experimental.cardinality(temp_ds).numpy()

if temp_batches < 2:
    raise ValueError("Dataset is too small for the requested validation/test split.")

val_batches = max(1, int(round(temp_batches * VAL_SPLIT / (VAL_SPLIT + TEST_SPLIT))))

val_ds = temp_ds.take(val_batches)
test_ds = temp_ds.skip(val_batches)

print("Approximate validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Approximate test batches:", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:

# Display sample images
plt.figure(figsize=(10, 8))

for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.tight_layout()
plt.show()



## Preprocessing and Performance Pipeline

ResNet-50 expects 224×224 RGB images. The `preprocess_input` function prepares the image values for the pre-trained ResNet-50 model.


In [ ]:

AUTOTUNE = tf.data.AUTOTUNE

def preprocess(images, labels):
    images = tf.cast(images, tf.float32)
    images = preprocess_input(images)
    return images, labels

train_prepared = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_prepared = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_prepared = test_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)



## Task B — Implementing Transfer Learning

A pre-trained ResNet-50 model is loaded without its original ImageNet classification head. A new classification layer is added according to the number of classes in the Kaggle dataset.

The pre-trained feature-extraction layers are frozen as required by the experiment.


In [ ]:

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax")
])

model.summary()


In [ ]:

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")



## Task C — Model Training

Train the transfer-learning model and record training accuracy, validation accuracy, training loss, and validation loss.


In [ ]:

EPOCHS = 10

history = model.fit(
    train_prepared,
    validation_data=val_prepared,
    epochs=EPOCHS
)

In [ ]:

# Record final training and validation metrics
final_train_acc = history.history["accuracy"][-1]
final_val_acc = history.history["val_accuracy"][-1]
final_train_loss = history.history["loss"][-1]
final_val_loss = history.history["val_loss"][-1]

print(f"Final Training Accuracy   : {final_train_acc:.4f}")
print(f"Final Validation Accuracy : {final_val_acc:.4f}")
print(f"Final Training Loss       : {final_train_loss:.4f}")
print(f"Final Validation Loss     : {final_val_loss:.4f}")


## Accuracy vs Epoch

In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epoch")
plt.legend()
plt.grid(True)
plt.show()


## Loss vs Epoch

In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epoch")
plt.legend()
plt.grid(True)
plt.show()


## Test Accuracy

In [ ]:

test_loss, test_accuracy = model.evaluate(test_prepared, verbose=1)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")


In [ ]:

# Display predictions from the transfer-learning model on test images
for images, labels in test_ds.take(1):
    raw_images = images.numpy().astype("uint8")
    processed_images = preprocess_input(tf.cast(images, tf.float32))
    probabilities = model.predict(processed_images, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    plt.figure(figsize=(12, 8))

    for i in range(min(9, len(raw_images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(raw_images[i])
        true_label = class_names[labels[i]]
        predicted_label = class_names[predictions[i]]
        plt.title(f"True: {true_label}\nPred: {predicted_label}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()



## Task D — Hugging Face Pre-trained Vision Model

The following section uses a pre-trained image-classification model available through Hugging Face Transformers.

The model is used for classification of sample images. Its labels may differ from the custom Kaggle dataset classes, so the comparison should focus on the predictions and observations rather than assuming the class names are identical.


In [ ]:

from transformers import pipeline

hf_classifier = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224"
)

print("Hugging Face vision model loaded successfully.")


In [ ]:

# Use a sample image from the test dataset.
# The Hugging Face pipeline accepts a PIL image.

from PIL import Image

sample_images = []

for images, labels in test_ds.take(1):
    for i in range(min(3, len(images))):
        arr = images[i].numpy().astype("uint8")
        sample_images.append(Image.fromarray(arr))

for i, image in enumerate(sample_images):
    results = hf_classifier(image, top_k=5)

    print(f"\nSample Image {i + 1}")
    for result in results:
        print(f"{result['label']}: {result['score']:.4f}")



## Compare Transfer Learning and Hugging Face Predictions

Use the following cell to view both models' predictions for the same sample images. Note that the ResNet-50 model was trained specifically on the Kaggle dataset, whereas the Hugging Face ViT model uses its own pre-trained label space.


In [ ]:

# Compare predictions for the same sample images

for i, image in enumerate(sample_images):
    image_array = np.array(image)

    # Transfer-learning prediction
    x = np.expand_dims(image_array.astype(np.float32), axis=0)
    x = preprocess_input(x)
    tl_probs = model.predict(x, verbose=0)[0]
    tl_idx = int(np.argmax(tl_probs))
    tl_label = class_names[tl_idx]
    tl_score = float(tl_probs[tl_idx])

    # Hugging Face prediction
    hf_result = hf_classifier(image, top_k=1)[0]

    print(f"Sample {i + 1}")
    print(f"Transfer Learning (ResNet-50): {tl_label} ({tl_score:.4f})")
    print(f"Hugging Face ViT             : {hf_result['label']} ({hf_result['score']:.4f})")
    print("-" * 60)
